## Data Ingestion (Streaming)

Here, we store the streaming data coming from **Kafka** as aggregated data in our Landing Zone.

**Importing Useful Libraries**

In [1]:
from kafka import KafkaConsumer
from dotenv import load_dotenv
import json
import boto3
import io
import os
import time

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

# Deserializer function
def deserialize(m):
    return json.loads(m.decode("utf-8"))

In [2]:
# -------------------------
# Kafka Consumers
# -------------------------
consumer_weather = KafkaConsumer(
    'weather-barcelona',
    group_id='weather-group',
    bootstrap_servers='kafka:9092',
    value_deserializer=deserialize,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
)

consumer_air = KafkaConsumer(
    'airquality-barcelona',
    group_id='airquality-group',
    bootstrap_servers='kafka:9092',
    value_deserializer=deserialize,
    auto_offset_reset='earliest',
    enable_auto_commit=False,
)

In [3]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

In [4]:
# -------------------------
# Aggregation loop
# -------------------------
bucket_name = 'landing-zone'

# Fetch new messages given a consumer
def fetch_new_messages(consumer):
    records = consumer.poll(timeout_ms=3000)

    rows = []
    for tp, msgs in records.items():
        for msg in msgs:
            rows.append(msg.value)

    return rows

# Load an existing file
def load_existing(bucket_name, key):
    try:
        obj = s3.get_object(Bucket=bucket_name, Key=key)
        return json.loads(obj['Body'].read())
    except s3.exceptions.NoSuchKey:
        return []

# Aggregated file paths
weather_agg_key = 'persistent-landing/semistructured/weather-barcelona.json'
air_agg_key = 'persistent-landing/semistructured/airquality-barcelona.json'

while True:

    # ----- Weather -----
    weather_new = fetch_new_messages(consumer_weather)
    weather_existing = load_existing(bucket_name, weather_agg_key)

    weather_updated = weather_existing + weather_new

    s3.put_object(
        Bucket=bucket_name,
        Key=weather_agg_key,
        Body=json.dumps(weather_updated)
    )

    print(f"Updated weather aggregated file ({len(weather_new)} new records)")

    # ----- Air Quality -----
    air_new = fetch_new_messages(consumer_air)
    air_existing = load_existing(bucket_name, air_agg_key)

    air_updated = air_existing + air_new

    s3.put_object(
        Bucket=bucket_name,
        Key=air_agg_key,
        Body=json.dumps(air_updated)
    )

    print(f"Updated air quality aggregated file ({len(air_new)} new records)")

    # Commit offsets AFTER successful processing
    consumer_weather.commit()
    consumer_air.commit()

    # Wait before next batch
    time.sleep(60)

Updated weather aggregated file (4 new records)
Updated air quality aggregated file (4 new records)
Updated weather aggregated file (1 new records)
Updated air quality aggregated file (1 new records)


KeyboardInterrupt: 